# kaggle-vllm Milestone 2 — Qwen TP1/TP2 Concurrency Crossover

This notebook executes the real **Milestone 2: High Concurrency Crossover & VRAM Saturation** experiment for `kaggle-vllm` on a fresh Kaggle **GPU T4 ×2** session.

The experiment compares `tensor_parallel_size=1` and `tensor_parallel_size=2` for the same pinned `Qwen/Qwen2.5-3B-Instruct` Transformers snapshot across request concurrency:

`1, 4, 8, 16, 32, 64`

The notebook records output throughput, p95 TTFT, p95 TPOT, request success/failure evidence, GPU telemetry, vLLM metrics, topology, environment identity, and SHA256 checksums. It tests whether a throughput or capacity crossover occurs; it does **not** assume TP2 must win.

**Required Kaggle settings**

- Accelerator: **GPU T4 ×2**
- Internet: **On**, unless the pinned Transformers model is attached under `/kaggle/input`
- Optional Kaggle Secret: `HF_TOKEN`
- Run from a fresh Kaggle session


## 1. Pin the reviewed source, model, and evidence paths

This cell defines the immutable identities used by the experiment.

- `REVIEWED_SOURCE_COMMIT` identifies the exact reviewed Milestone 2 implementation.
- `MODEL_REVISION` pins the exact Qwen snapshot used for both TP1 and TP2.
- Evidence is written only under `/kaggle/working`.
- Existing evidence or ZIP paths are refused to prevent accidental overwrite.


In [ ]:
import hashlib, json, os, platform, shutil, subprocess, sys, tarfile, urllib.request
from pathlib import Path

REVIEWED_SOURCE_COMMIT = "4f8dcc1c032d65d54b1cce3ca213535d68fd5099"
MODEL_REPOSITORY = "Qwen/Qwen2.5-3B-Instruct"
MODEL_REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"

# Set to a read-only /kaggle/input/... Transformers snapshot to run offline.
ATTACHED_MODEL_PATH = None

WORK_ROOT = Path("/kaggle/working/kaggle-vllm-milestone-2-work")
EVIDENCE_DIR = Path("/kaggle/working/kaggle-vllm-milestone-2")
EVIDENCE_ZIP = Path("/kaggle/working/kaggle-vllm-milestone-2.zip")
SOURCE_URL = (
    f"https://github.com/kaggle-vllm/kaggle-vllm/archive/"
    f"{REVIEWED_SOURCE_COMMIT}.tar.gz"
)

print("Reviewed source commit:", REVIEWED_SOURCE_COMMIT)
print("Pinned model:", MODEL_REPOSITORY, MODEL_REVISION)
print("Python:", sys.version)
print("Platform:", platform.platform())

assert Path("/kaggle/working").is_dir(), "Run this notebook on Kaggle"
assert not EVIDENCE_DIR.exists(), f"Refusing existing evidence: {EVIDENCE_DIR}"
assert not EVIDENCE_ZIP.exists(), f"Refusing existing ZIP: {EVIDENCE_ZIP}"


## 2. Verify the Kaggle dual-T4 hardware contract

Milestone 2 is valid only for the documented Kaggle profile. This cell checks:

- PyTorch `2.10.0+cu128`
- CUDA runtime `12.8`
- exactly two visible GPUs
- both GPUs are Tesla T4
- both GPUs report compute capability 7.5 / SM75

It also records `nvidia-smi` and the observed GPU topology. If these assertions fail, do not force the experiment to continue.


In [ ]:
import torch

print("Torch:", torch.__version__, torch.__file__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

for command in (["nvidia-smi"], ["nvidia-smi", "topo", "-m"]):
    print("$", " ".join(command), flush=True)
    subprocess.run(command, check=True)

assert torch.__version__ == "2.10.0+cu128"
assert torch.version.cuda == "12.8"
assert torch.cuda.device_count() == 2
assert all("Tesla T4" in torch.cuda.get_device_name(i) for i in range(2))
assert all(torch.cuda.get_device_capability(i) == (7, 5) for i in range(2))

print("PASS: expected Kaggle T4 x2 hardware profile observed.")


## 3. Download and stage the exact reviewed kaggle-vllm source

The notebook downloads the source archive for the reviewed Git commit, performs a safe extraction, builds the lightweight SDK wheel, and installs that SDK into an isolated target directory.

This does **not** rebuild the native CUDA/vLLM runtime. It only stages the Python SDK whose benchmark runner and evidence schema are under test.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def safe_extract(archive, destination):
    destination = Path(destination).resolve()
    with tarfile.open(archive, "r:gz") as bundle:
        for member in bundle.getmembers():
            if member.issym() or member.islnk():
                raise RuntimeError(f"Refusing archive link: {member.name}")
            target = (destination / member.name).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Unsafe archive member: {member.name}")
        bundle.extractall(destination, filter="data")

if WORK_ROOT.exists():
    resolved = WORK_ROOT.resolve()
    assert resolved.parent == Path("/kaggle/working")
    assert not WORK_ROOT.is_symlink()
    shutil.rmtree(WORK_ROOT)

SOURCE_UNPACK = WORK_ROOT / "source"
SDK_DIST = WORK_ROOT / "dist"
SDK_TARGET = WORK_ROOT / "sdk"
SOURCE_ARCHIVE = WORK_ROOT / "reviewed-source.tar.gz"

SOURCE_UNPACK.mkdir(parents=True)
SDK_DIST.mkdir()

urllib.request.urlretrieve(SOURCE_URL, SOURCE_ARCHIVE)
print("Source archive SHA256:", sha256_file(SOURCE_ARCHIVE))

safe_extract(SOURCE_ARCHIVE, SOURCE_UNPACK)
roots = [path for path in SOURCE_UNPACK.iterdir() if path.is_dir()]
assert len(roots) == 1
SOURCE_ROOT = roots[0]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "build>=1.2"],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "build",
        "--wheel",
        "--outdir",
        str(SDK_DIST),
        str(SOURCE_ROOT),
    ],
    check=True,
)

wheels = sorted(SDK_DIST.glob("kaggle_vllm-*.whl"))
assert len(wheels) == 1

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--target",
        str(SDK_TARGET),
        str(wheels[0]),
    ],
    check=True,
)

sys.path.insert(0, str(SDK_TARGET))

SDK_ENV = os.environ.copy()
SDK_ENV["PYTHONPATH"] = str(SDK_TARGET)

print("SOURCE_ROOT:", SOURCE_ROOT)
print("SDK wheel SHA256:", sha256_file(wheels[0]))


## 4. Bootstrap the immutable native vLLM runtime and run strict doctor

This stage resolves the previously validated native vLLM CUDA wheel, verifies its SHA256, activates the staged runtime without replacing Kaggle's system PyTorch/CUDA stack, and runs `kaggle-vllm doctor --strict`.

A strict-doctor failure is a stop condition. The benchmark should not continue on an environment outside the validated compatibility contract.


In [ ]:
from kaggle_vllm.bootstrap import activate_runtime, bootstrap
from kaggle_vllm.doctor import run_doctor
from kaggle_vllm.environment import as_json

bootstrap_result = bootstrap(strict=True)
activate_runtime(bootstrap_result.manifest)

assert run_doctor(strict=True) == 0

print(as_json())
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvidia-smi", "topo", "-m"], check=True)

# Preserve the runtime environment that activation added.
SDK_ENV.update(os.environ)

print("Immutable native runtime bootstrap and strict doctor: PASS")


## 5. Resolve one identical pinned Transformers model for TP1 and TP2

For scientific comparability, both tensor-parallel configurations use the same Qwen model weights and revision.

The existing topology-specific TP2 `sharded_state` artifact is deliberately not used here, because it cannot serve as the TP1 half of a fair comparison.

If Internet is enabled, the pinned snapshot is downloaded once into `/kaggle/working`. An attached read-only Transformers snapshot under `/kaggle/input` may be used instead.


In [ ]:
if ATTACHED_MODEL_PATH is not None:
    MODEL_PATH = Path(ATTACHED_MODEL_PATH).resolve()
    assert Path("/kaggle/input") in MODEL_PATH.parents
    print("Using attached read-only Transformers model:", MODEL_PATH)
else:
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "huggingface_hub>=0.36",
            ],
            check=True,
        )
        from huggingface_hub import snapshot_download

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
        print("HF_TOKEN available:", bool(token))
    except Exception as error:
        token = None
        print("HF_TOKEN available: False (", type(error).__name__, ")")

    MODEL_PATH = Path(
        snapshot_download(
            repo_id=MODEL_REPOSITORY,
            revision=MODEL_REVISION,
            token=token,
            cache_dir=str(WORK_ROOT / "model-cache"),
        )
    ).resolve()

print("Resolved model path:", MODEL_PATH)

assert (MODEL_PATH / "config.json").is_file()
assert not any(
    MODEL_PATH.glob("model-rank-*-part-*.safetensors")
), "TP-specific sharded_state is forbidden"


## 6. Print the side-effect-free 12-cell benchmark plan

Before allocating GPUs for the real sweep, this cell asks the Milestone 2 runner to print its complete plan in dry-run mode.

The runner environment explicitly prepends `SOURCE_ROOT/src` to `PYTHONPATH`. This fixes the orchestration issue discovered during the first live execution, where the child process could not import `kaggle_vllm`.

The dry run must not create evidence, launch vLLM, or mutate the model/runtime.


In [ ]:
RUNNER = SOURCE_ROOT / "scripts" / "kaggle_concurrency_crossover.py"

COMMON = [
    "--model",
    str(MODEL_PATH),
    "--model-source",
    "local_transformers",
    "--model-revision",
    MODEL_REVISION,
    "--output-dir",
    str(EVIDENCE_DIR),
]

RUNNER_ENV = dict(SDK_ENV)
SDK_SRC = SOURCE_ROOT / "src"

assert RUNNER.is_file(), f"Runner not found: {RUNNER}"
assert SDK_SRC.is_dir(), f"SDK src directory not found: {SDK_SRC}"
assert (SDK_SRC / "kaggle_vllm" / "__init__.py").is_file()

existing_pythonpath = RUNNER_ENV.get("PYTHONPATH", "")
RUNNER_ENV["PYTHONPATH"] = (
    str(SDK_SRC)
    if not existing_pythonpath
    else str(SDK_SRC) + os.pathsep + existing_pythonpath
)

print("Runner:", RUNNER)
print("SDK source:", SDK_SRC)
print("PYTHONPATH:", RUNNER_ENV["PYTHONPATH"])
print("Side-effect-free benchmark plan:")

subprocess.run(
    [
        sys.executable,
        str(RUNNER),
        "--dry-run",
        *COMMON,
    ],
    check=True,
    env=RUNNER_ENV,
    cwd=str(SOURCE_ROOT),
)


## 7. Execute the real TP1/TP2 concurrency matrix

This is the expensive GPU stage.

The runner executes the interleaved matrix:

- TP1 / TP2 at concurrency 1
- TP1 / TP2 at concurrency 4
- TP1 / TP2 at concurrency 8
- TP1 / TP2 at concurrency 16
- TP1 / TP2 at concurrency 32
- TP1 / TP2 at concurrency 64

Each benchmark cell uses a fresh server lifecycle and writes its own JSON result, server log, Prometheus metrics snapshot, request ledger, and GPU telemetry.

The notebook then verifies that all twelve evidence sets and required top-level files are present.


In [ ]:
print("Executing Milestone 2 benchmark")
print("Reviewed source:", REVIEWED_SOURCE_COMMIT)
print("Runner:", RUNNER)
print("SDK source:", SDK_SRC)
print("Model:", MODEL_PATH)
print("Model revision:", MODEL_REVISION)
print("Evidence directory:", EVIDENCE_DIR)
print("PYTHONPATH:", RUNNER_ENV["PYTHONPATH"])

command = [
    sys.executable,
    str(RUNNER),
    "--source-identity",
    REVIEWED_SOURCE_COMMIT,
    *COMMON,
]

print("\nCommand:")
print(" ".join(map(str, command)))
print("\nStarting real TP1/TP2 concurrency matrix...\n")

subprocess.run(
    command,
    check=True,
    env=RUNNER_ENV,
    cwd=str(SOURCE_ROOT),
)

required = [
    "run-metadata.json",
    "environment.json",
    "topology.txt",
    "summary.json",
    "SHA256SUMS.txt",
]

missing = [
    name
    for name in required
    if not (EVIDENCE_DIR / name).is_file()
]

assert not missing, f"Missing required top-level evidence: {missing}"

missing_cell_files = []

for tp in (1, 2):
    for concurrency in (1, 4, 8, 16, 32, 64):
        stem = f"qwen-tp{tp}-c{concurrency:02d}"

        expected = [
            EVIDENCE_DIR / f"{stem}.json",
            EVIDENCE_DIR / f"{stem}.server.log",
            EVIDENCE_DIR / f"{stem}.metrics.txt",
            EVIDENCE_DIR / f"{stem}.telemetry.jsonl",
            EVIDENCE_DIR / f"{stem}-requests.jsonl",
        ]

        for path in expected:
            if not path.is_file():
                missing_cell_files.append(str(path))

assert not missing_cell_files, (
    "Missing per-cell evidence:\n" + "\n".join(missing_cell_files)
)

summary = json.loads((EVIDENCE_DIR / "summary.json").read_text())

print("\n=== Milestone 2 summary ===")
print(json.dumps(summary, indent=2))

print("\nPASS: all required Milestone 2 evidence files are present.")


## 8. Verify the evidence manifest and create the transfer ZIP

This cell verifies every file listed in `SHA256SUMS.txt` before creating the ZIP.

The ZIP is the primary transfer artifact for independent review. Its SHA256 must be recorded before the Kaggle session is stopped, and the same hash should be verified again after downloading to the local workstation.

Passing this cell means the bundle is internally consistent; it does not by itself prove a scientific crossover claim.


In [ ]:
assert EVIDENCE_DIR.is_dir(), f"Evidence directory missing: {EVIDENCE_DIR}"

manifest = EVIDENCE_DIR / "SHA256SUMS.txt"
assert manifest.is_file(), f"Checksum manifest missing: {manifest}"

print("Verifying internal evidence checksums...")

verification = subprocess.run(
    ["sha256sum", "-c", str(manifest)],
    cwd=str(EVIDENCE_DIR),
    text=True,
    capture_output=True,
)

print(verification.stdout)

if verification.stderr:
    print(verification.stderr)

assert verification.returncode == 0, (
    "Internal evidence checksum verification FAILED"
)

print("PASS: internal SHA256SUMS.txt verification succeeded.")

assert not EVIDENCE_ZIP.exists(), (
    f"Refusing to overwrite existing ZIP: {EVIDENCE_ZIP}"
)

archive_base = str(EVIDENCE_ZIP.with_suffix(""))

created = Path(
    shutil.make_archive(
        archive_base,
        "zip",
        root_dir=EVIDENCE_DIR.parent,
        base_dir=EVIDENCE_DIR.name,
    )
)

assert created.resolve() == EVIDENCE_ZIP.resolve()

zip_sha256 = sha256_file(EVIDENCE_ZIP)

print("\nEvidence ZIP:", EVIDENCE_ZIP)
print("Evidence ZIP size:", EVIDENCE_ZIP.stat().st_size, "bytes")
print("Evidence ZIP SHA256:", zip_sha256)

print(
    "\nGPU execution bundle created successfully. "
    "Milestone 2 acceptance is still pending independent evidence review."
)


## 9. Read-only scientific result summary

This final inspection cell does not modify evidence.

It prints:

- the complete `summary.json`
- per-cell status, successes, failures, output throughput, p95 TTFT, and p95 TPOT
- the full evidence-file inventory
- the final ZIP SHA256

Interpretation rules:

- high GPU memory usage alone is not OOM
- timeout alone is not CUDA OOM
- throughput slowdown alone does not prove KV-cache exhaustion
- `null` crossover fields are valid results
- throughput crossover and capacity crossover must remain distinct


In [ ]:
ROOT = EVIDENCE_DIR
SUMMARY_PATH = ROOT / "summary.json"

assert SUMMARY_PATH.is_file()

summary = json.loads(SUMMARY_PATH.read_text())

print("=== RAW SUMMARY ===")
print(json.dumps(summary, indent=2))

print("\n=== COMPACT CELL TABLE ===")
print(
    f"{'TP':<4}"
    f"{'C':<6}"
    f"{'STATUS':<14}"
    f"{'SUCCESS':<10}"
    f"{'FAILED':<9}"
    f"{'TOK/S':<14}"
    f"{'TTFT P95':<14}"
    f"{'TPOT P95':<14}"
)
print("-" * 95)

for concurrency in (1, 4, 8, 16, 32, 64):
    for tp in (1, 2):
        path = ROOT / f"qwen-tp{tp}-c{concurrency:02d}.json"
        data = json.loads(path.read_text())

        measurements = data.get("measurements", {})
        ttft = measurements.get("ttft_seconds") or {}
        tpot = measurements.get("tpot_seconds") or {}

        print(
            f"{tp:<4}"
            f"{concurrency:<6}"
            f"{str(data.get('status')):<14}"
            f"{str(measurements.get('successful_requests')):<10}"
            f"{str(measurements.get('failed_requests')):<9}"
            f"{str(measurements.get('output_throughput_tokens_per_second')):<14}"
            f"{str(ttft.get('p95')):<14}"
            f"{str(tpot.get('p95')):<14}"
        )

print("\n=== EVIDENCE INVENTORY ===")
for path in sorted(ROOT.iterdir()):
    if path.is_file():
        print(path.name, path.stat().st_size)

print("\n=== TRANSFER ARTIFACT ===")
print("ZIP:", EVIDENCE_ZIP)
print("ZIP bytes:", EVIDENCE_ZIP.stat().st_size)
print("ZIP SHA256:", sha256_file(EVIDENCE_ZIP))

print(
    "\nExecution is complete. Scientific acceptance should be based on "
    "the reviewed JSON/log/metrics/telemetry evidence, not screenshots alone."
)
